In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import coo_matrix
from implicit.als import AlternatingLeastSquares

In [2]:
train = pd.read_csv('../dataset/train_A.tsv', sep='\t')
test = pd.read_csv('../dataset/test.tsv', sep='\t')
test_A = test[test['user_id'].str.endswith('_A')]


In [3]:
# 関連度スコアの計算
def compute_relevance(row):
    if row["event_type"] == 3 and row["ad"] == 1:  # 広告経由の購入
        return 3
    elif row["event_type"] == 2:  # 広告クリック
        return 2
    elif row["event_type"] == 1:  # 詳細ページ閲覧
        return 1
    else:  # カート追加
        return 0


In [4]:
# 関連度スコアの設定
train['relevance'] = train.apply(compute_relevance, axis=1)


In [ ]:
train = train[train['relevance'] > 0]  # 0を除く（重要）


In [6]:
# 基準日（学習データの最終日）
end_date = pd.to_datetime("2017-04-30")

# time_stampをdatetime型に変換し、日付を抽出
train['date'] = pd.to_datetime(train['time_stamp']).dt.date
train['date'] = pd.to_datetime(train['date'])

# 減衰率を計算
train[f'weight_r_0.9'] = train['date'].apply(lambda x: 0.9 ** (end_date - x).days)


In [7]:
# 関連度 * 減衰率でスコア作成
train['score'] = train['relevance'] * train['weight_r_0.9']


In [8]:
# ユーザー・アイテムに整数IDを振る
user_ids = train['user_id'].astype('category')
item_ids = train['product_id'].astype('category')

train['user_idx'] = user_ids.cat.codes
train['item_idx'] = item_ids.cat.codes


In [9]:
# スパース行列（item-user行列：implicitはitemが行、userが列）
item_user_data = coo_matrix(
    (train['score'].astype(float), (train['item_idx'], train['user_idx']))
)


In [10]:
# ALSモデルの訓練
model = AlternatingLeastSquares(factors=50, regularization=0.01, iterations=15)
model.fit(item_user_data)


/home/mayrin/develop/venv/lib/python3.12/site-packages/implicit/cpu/als.py:95: RuntimeWarning: OpenBLAS is configured to use 16 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'OPENBLAS_NUM_THREADS=1' or by calling 'threadpoolctl.threadpool_limits(1, "blas")'. Having OpenBLAS use a threadpool can lead to severe performance issues here.
  check_blas_config()
/home/mayrin/develop/venv/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed coo_matrix instead. Converting to CSR took 0.05533480644226074 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

In [11]:
# ユーザー → Top-22商品の推薦
user_idx_to_id = dict(enumerate(user_ids.cat.categories))
item_idx_to_id = dict(enumerate(item_ids.cat.categories))

topn = 22
recommendations = []

for user_idx in range(len(user_ids.cat.categories)):
    recommended = model.recommend(user_idx, item_user_data.T, N=topn)

    for rank, (item_idx, score) in enumerate(recommended, start=1):
        recommendations.append({
            'user_id': user_idx_to_id[user_idx],
            'product_id': item_idx_to_id[item_idx],
            'score': score,
            'rank': rank
        })

recommend_df = pd.DataFrame(recommendations)


ValueError: user_items needs to be a CSR sparse matrix

In [ ]:
pd.DataFrame(recommend_df, columns=['user_id', 'product_id', 'rank']).to_csv('../dataset/implicit_A.tsv', sep='\t', index=False)